<a href="https://colab.research.google.com/github/ricardoasf/Previsao-Series-Temporais/blob/main/MA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Definição das bibliotecas usadas
import pandas as pd #álgebra linear
import numpy as np #processamento de dados

import matplotlib as mpl
import matplotlib.pyplot as plt #visualização de dados
import seaborn as sns #visualização de dados

############### PARAMETRIZAÇÃO PARA EXECUÇÃO #############
# 1 - para Jupyter/Outras IDEs;
# 2 - para GoogleColab
carga_dados = 2
##########################################################

if carga_dados == 1:
    #Carrega Série Temporal a partir de Planilha de Dados no formato de DataFrame
    import tkinter as tk
    from tkinter import filedialog
    root = tk.Tk()
    root.withdraw()
    caminho_arquivo = filedialog.askopenfilename()
    df = pd.read_csv(caminho_arquivo)
else:
    #Carrega Dados para GoogleColab
    import io
    from google.colab import files
    uploaded = files.upload()
    df = pd.read_csv(io.BytesIO(uploaded['Dataset1.csv']))

df

In [ ]:
df['Preco'].describe()

In [ ]:
df['DateTime'] = pd.to_datetime(df['Data'])
df

In [ ]:
plt.figure(figsize=(15,5))
plt.plot(df['DateTime'], df['Preco'])
plt.ylabel('Preço')
plt.xlabel('Data e Hora')
plt.show()

In [ ]:
# Média Móvel como Suavização de Dados
tamanho_janela = 42
df['MA_Suavizacao'] = df['Preco'].rolling(tamanho_janela).mean()

plt.figure(figsize=(15,5))
plt.plot(df['DateTime'], df['Preco'])
plt.plot(df['DateTime'], df['MA_Suavizacao'], 'r')
plt.ylabel('Preço')
plt.xlabel('Data e Hora')
plt.show()

In [ ]:
#Torna a Coluna DateTime o índice do DataFrame
df = df.set_index(df['DateTime'])

#Clona o DataFrame e Normaliza as Colunas pela primeira amostra
df_clone = df[:]
df_clone['Preco'] = df['Preco'] / df['Preco'].iloc[0]

#Copia apenas as colunas de interesse do Clone para o novo DataFrame
df_norm = df_clone[['Preco']]

#Clona o DataFrame normalizado para um novo
df_lagged = df_norm[:]

#Determina os atrasos (em amostras)
lag = 1

#Gera o DataFrame com as Colunas de Atraso
#df_lagged = df_norm.assign(**{f'{col}_t-{lag}': df_norm[col].shift(lag) for lag in lags for col in df_norm})
df_lagged = df_norm.assign(**{f'{col}_t-{lag}': df_norm[col].shift(lag) for col in df_norm})

#Exclui valor NaN
df_proc = df_lagged.dropna()

#Mostra dados mais recentes
df_proc

In [ ]:
#Define as colunas que serão variáveis de entrada - explicativas
features = df_proc.columns[[1]]
features

In [ ]:
#Define as coluna que será variável de saída - interesse
target = df_proc.columns[[0]]
target

In [ ]:
#Separação de Dados para Ajuste e Validação dos Modelos
df_t_ajuste = df_proc['2009-07-01 00:00:00':'2009-07-31 23:00:00'].index.values
df_t_validacao = df_proc['2009-08-01 00:00:00':].index.values
df_dados_ajuste = df_proc['2009-07-01 00:00:00':'2009-07-31 23:00:00']
df_dados_validacao = df_proc['2009-08-01 00:00:00':]

X_ajuste = df_dados_ajuste[features].values
Y_ajuste = df_dados_ajuste[target].values
X_validacao = df_dados_validacao[features].values
Y_validacao = df_dados_validacao[target].values

In [ ]:
# Parametrização de Modelo MA
from statsmodels.tsa.arima.model import ARIMA
MA_model = ARIMA(X_ajuste, order=(0, 0, 5)) #order = (p,d,q)
MA_param = MA_model.fit()

In [ ]:
df_previsto_MA = MA_param.predict(start=len(X_validacao), end=len(X_validacao)+len(X_validacao)-1)
df_previsto_MA_denorm = df_previsto_MA * df['Preco'].iloc[0]

In [ ]:
plt.figure(figsize=(15,5))
plt.plot(df['DateTime'], df['Preco'])
plt.plot(df_t_validacao, df_previsto_MA_denorm, 'r')
plt.ylabel('Preco')
plt.xlabel('Data e Hora')
plt.show()